# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring a FAIR^2 Clinical Oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields/columns by their `@id`.

We display all record sets and their fields, using the `@id` as the main reference.

In [ ]:
# List all record sets (referenced by their @id)
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = [rs for rs in dataset.record_sets()]

if not record_sets:
    # alternative method if 'metadata.record_sets' is not available
    record_sets = list(dataset._schema.record_sets.keys())

print('Available record sets:')
for rs in record_sets:
    if hasattr(rs, '@id'):
        rs_id = rs['@id'] if isinstance(rs, dict) else rs.@id
    else:
        rs_id = str(rs)
    print(f"  Record set @id: {rs_id}")
    # For each record set, list fields/columns by @id
    try:
        schema_rs = dataset.record_set(rs_id)
        print("    Fields:")
        if hasattr(schema_rs, 'fields'):
            for field in schema_rs.fields:
                print(f"      - {getattr(field, '@id', getattr(field, 'id', field))}")
        elif hasattr(schema_rs, 'columns'):
            for column in schema_rs.columns:
                print(f"      - {getattr(column, '@id', getattr(column, 'id', column))}")
    except Exception as ex:
        print(f"    Could not read fields for record set {rs_id}: {ex}")

## 3. Data Extraction
Load all records from one or more record sets into pandas DataFrames for analysis.
Use only `@id` for references.

In [ ]:
# --- RECORD SET ID SETUP ---
# Inspect the available record sets using the output above.
# We'll use the main record set, which by Croissant convention is often the same as the dataset URL, or the main data table.
# For this dataset, we'll try loading all available record sets.

main_record_sets = []

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        main_record_sets.append(getattr(rs, '@id', getattr(rs, 'id', None)))
elif 'recordSet' in metadata.__dict__ and metadata.recordSet:
    main_record_sets = metadata.recordSet
elif hasattr(dataset._schema, 'record_sets'):
    main_record_sets = list(dataset._schema.record_sets.keys())

# Remove None values and ensure unique
main_record_sets = [rs for rs in main_record_sets if rs]
main_record_sets = list(set(main_record_sets))

if not main_record_sets:
    # as fallback, try commonly structured @id
    # Attempt to infer from schema URL and dataset title
    main_record_sets = [dataset._schema.data['@id']]

print('Will load these record sets:')
print(main_record_sets)

dataframes = {}
for record_set_id in main_record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record set {record_set_id} loaded, shape: {df.shape}")
        print(f"Columns: {list(df.columns)}\n")
    except Exception as ex:
        print(f"Could not load records for {record_set_id}: {ex}")

# Show head for the first valid DataFrame
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"First few rows of record set {first_id}:")
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply preprocessing steps such as filtering, normalization, and grouping by columns (referenced by their @id).

In [ ]:
from IPython.display import display

# Select the main DataFrame for EDA
record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]

# Show available columns
print("Available columns (@id):")
print(list(df.columns))

# Example: Pick a numeric field by its @id (commonly named e.g. 'Age' or similar — check actual columns)
# Let's attempt to find a numeric column (will default to 'Age' if present, else pick the first float/int)
import numpy as np
numeric_field_id = None
for col in df.columns:
    if df[col].dtype in [np.int64, np.float64]:
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try a common field name
    for col in df.columns:
        if 'age' in col.lower():
            numeric_field_id = col
            break

print(f"Selected numeric field for analysis: {numeric_field_id}")

if numeric_field_id is not None:
    # Use the median as a threshold for filtering
    if df[numeric_field_id].dtype in [np.float64, np.int64]:
        threshold = df[numeric_field_id].median()
    elif df[numeric_field_id].dtype == object:
        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = df[numeric_field_id].median()
        except Exception:
            threshold = 0
            print("Warning: Could not automatically select threshold. Using 0.")
    else:
        threshold = 0

    # Filtered DataFrame based on threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a possible categorical field (@id)
    # Try a column with 'Sex', 'gender', or 'MSI', or anatomical location
    group_field_candidates = [
        col for col in df.columns if any(t in col.lower() for t in ['sex', 'gender', 'msi', 'location', 'site', 'type'])
    ]
    group_field = group_field_candidates[0] if group_field_candidates else None

    if group_field:
        print(f"Grouping by field: {group_field}")
        # Only include columns that are numeric in the groupby
        numeric_cols = filtered_df.select_dtypes(include=[np.number]).columns.tolist()
        grouped_df = filtered_df.groupby(group_field)[numeric_cols].mean().reset_index()
        print("Grouped data (mean values per group):")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found to group.")
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize data distributions and field relationships. All columns are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
if numeric_field_id and numeric_field_id in df:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field is available, plot boxplots
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"Distribution of {numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric or group field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze a clinical–pathological oncology dataset published in FAIR^2 format with Croissant, using the `mlcroissant` library.

**Key steps and findings:**
- Inspected available record sets and columns using their `@id` fields.
- Transformed and filtered data for simple EDA.
- Visualized major numeric variables by group.

This FAIR^2 dataset enables transparent, reproducible oncology research and can be further used for advanced statistical modeling, predictive analysis, or integration with other croissant-formatted datasets.